# Langextract aplication with atributes

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import langextract as lx
from rich.pretty import pprint
import textwrap
import timeit

import re
import pandas as pd
from more_itertools import unique_justseen
from aymurai.api.endpoints.routers.misc.document_extract import extraction
from aymurai.database.utils import text_to_uuid

In [ ]:
MAIN_DF ='df-NER-vals-02-08-sin05.csv'
DOCS_PATH = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/' #'/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' 

In [ ]:
df = pd.read_csv(MAIN_DF)
df.head()

## Prompt & example definitions

In [ ]:
PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles para anonimización en documentos judiciales en español. Vas a leer cada parrafo con atención y extraer todas las entidades que correspondan según
las clases definidas más abajo. 

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como clase Persona, no confundir con siglas de otras cosas.
- Extraé SOLO spans EXACTOS que estén en el texto (no parafrasees ni infieras).
- Si una clase NO aparece, NO devuelvas nada de esa clase.
- NO inventes códigos ni números. No completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- NO son sensibles las siguientes entidades: 
                         1. La fecha y lugar de la resolucion del documento (por ej. "Buenos Aires, 29 de julio de 2022"), suele aparecen al comienzo del documento.
                         2. Información del juzgado como mail, direccion, telefono y cuenta de red social (por ej. "'Juzgado PCyF No 10 - Tacuarí 138, 7o Piso - juzcyf10@jusbaires.gob.ar - 4014-6821/20 - @jpcyf10'"), suele estar al pie del documento.
                         3. Las personas no sensibles como jueces, fiscales y secretarios.

- Si una entidad no está clara, NO la extraigas.
- Si una entidad está sutilmente mal escrita, incompleta, repetida o en un formato no estándar, pero detectas que corresponde a esa entendidad, extráela igual.

POSIBLES CLASES Y DESCRIPCIONES:
- BANCO: Debe especificar una entidad bancaria, 'Banco' no aplica como tal.
- CBU: número de 22 dígitos asociado a una entidad bancaria
- CORREO_ELECTRONICO: dirección de email, que no sea del juzgando.
- CUIT_CUIL: código único de identificación tributaria o laboral en Argentina (formato ##-########-#)
- CUIJ: código único de identificación judicial (formato ##-########-#)
- DIRECCION: puede presentarse como calle y altura (por ej. "9 de Julio 123"), intersección de calles (por ej. "Callao y Corrientes"), o número de domicilio 
- DNI: documento nacional de identidad de 7-8 dígitos, puede presentarse en el formato ##.###.### (numeros separados con puntos), la expresión "DNI" sin número no aplica como DNI.
- EDAD: edad de una persona, puede estar en años o meses
- ESTUDIOS: nivel educativo alcanzado (primario, secundario, terciario, universitario, posgrado, doctorado), puede estar acompañado de "incompleto", "completo", "finalizado", "en curso"
- FECHA: fechas en cualquier formato (dd/mm/aaaa, dd-mm-aaaa, dd de mes de aaaa, también puede ser dos fechas juntas como por ejemplo el 5 y 7 de mayo de 2020 y similares). No asignar como FECHA la fecha de resolución del documento, tampoco expresiones de fechas que no refieren a una fecha en particular, como por ejemplo, "en el día de ayer" o "a las 15:00 horas"
- LINK: URLs o enlaces web.
- LOC: nombres de localidades, provincias, países, continentes. NO asignar como LOC a lugares que no sean locaciones, como ser hospitales o referencias a lugares por nombres como "el domicilio de Olavarria"
- MARCA_AUTOMOVIL: marcas de automóviles (Ford, Chevrolet, Toyota, Renault, Fiat, etc)
- NACIONALIDAD: nacionalidades (argentina, italiana, española, uruguaya, chilena, paraguara, etc)
- NUM_CAJA_AHORRO: número de caja de ahorro o cuenta bancaria
- NUM_EXPEDIENTE: número de expediente judicial o administrativo en formato \d+/\d{4} (por ejemplo 1234/2020)
- NUM_MATRICULA: número de matrícula profesional (médica, abogacía, etc) o académica.
- PATENTE_DOMINIO: patentes o dominio de un vehículo. En Argentina, pueden ser de formato [A-Z]{3}\d{3} o [A-Z]{2}\d{3}[A-Z]{2}
- PER: Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.                     
- NUM_ACTUACION: Número identificatorio de una actuación administrativa o contravencional.
- TELEFONO: Número telefónico (fijo o celular).
                         
Razona detenidamente antes de elegir casa entidad y asociá dicho razonamiento a una variable justification y score de confianza de tu razonamiento, luego, para todo el parrafo crea una 
salida que sea una lista de diccionarios, uno por clase extraida, con las siguientes claves:
- text: el texto exacto de la entidad
- label: la clase de la entidad (una de las listadas más arriba)
""")


In [ ]:
# -----------------
# Ejemplos balanceados (judiciales)
#   1) PER/FECHA/DIRECCION/LOC
#   2) Un único ejemplo de códigos (para enseñar formato)
#   3) Datos personales típicos de actuaciones
#   4) NEGATIVO: no hay códigos -> salida vacía
# -----------------
examples = [
    lx.data.ExampleData(
        text=textwrap.dedent("""En la Ciudad Autónoma de Buenos Aires, el día 5 de mayo de 2023, "
              "el Sr. Fiscal hace saber que Juan Pérez se domicilia en la calle "
              "Sarmiento 1234, localidad de Moreno."""),
        extractions=[
            lx.data.Extraction(extraction_class="FECHA",     extraction_text="5 de mayo de 2023"),
            lx.data.Extraction(extraction_class="PER",       extraction_text="Juan Pérez",attributes={'ROL':'Acusado'},group_index=1),
            lx.data.Extraction(extraction_class="DIRECCION", extraction_text="Sarmiento 1234",group_index=1),
            lx.data.Extraction(extraction_class="LOC",       extraction_text="Moreno",group_index=1),
        ],
    ),

    lx.data.ExampleData(
        text = textwrap.dedent(""""3) Abstenerse de ingresar y/o concurrir a la Villa 13"""),
        extractions = [
            lx.data.Extraction(extraction_class="LOC", extraction_text = "Villa 13")
        ]
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""JUZGADO NACIONAL EN LO CRIMINAL Y CORRECCIONAL N° 10 - Secretaría N° 19. "
              "Causa N° 52345/2022. CUIJ: 12-34567890-1. Actuación N° 2022-009876."""),
        extractions=[
            lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"),
            lx.data.Extraction(extraction_class="CUIJ",           extraction_text="12-34567890-1"),
            lx.data.Extraction(extraction_class="NUM_ACTUACION",  extraction_text="2022-009876"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""Comparece Miguel Torres, DNI 30123456, de 34 años de edad, nacionalidad paraguaya, "
              "con estudios secundarios completos, con último domicilio en Av. Corrientes 3456 de esta ciudad, "
              "junto a su cuñado Jorge Pérez."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Miguel Torres"),
            lx.data.Extraction(extraction_class="DNI",         extraction_text="30123456"),
            lx.data.Extraction(extraction_class="EDAD",        extraction_text="34"),
            lx.data.Extraction(extraction_class="NACIONALIDAD",extraction_text="paraguaya"),
            lx.data.Extraction(extraction_class="ESTUDIOS",    extraction_text="estudios secundarios completos"),
            lx.data.Extraction(extraction_class="DIRECCION",   extraction_text="Av. Corrientes 3456"),
            lx.data.Extraction(extraction_class="PER",         extraction_text="Jorge Pérez"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""1) transferencia defondos a cuenta de terceros por $167.000,- hacia una cuenta a nombre de la Sra. Carla Analía Gonzales, CUIL 27-25011757-0, CBU 0740399088000036512321, del Banco Santander. """),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Carla Analía Gonzales"),
            lx.data.Extraction(extraction_class="CUIL",       extraction_text="27-25011757-0"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0740399088000036512321"),
            lx.data.Extraction(extraction_class="BANCO",      extraction_text="Banco Santander"),
        ],
    ),

  lx.data.ExampleData(
        text=textwrap.dedent("""teléfono celular 1141504528 y dirección de correo electrónico alejandro.perezgarcia@gmail.com."""),
        extractions=[
            lx.data.Extraction(extraction_class="TELEFONO",     extraction_text="1141504528"),
            lx.data.Extraction(extraction_class="CORREO_ELECTRONICO", extraction_text="alejandro.perezgarcia@gmail.com"),
        ],
    ),
lx.data.ExampleData(
        text=textwrap.dedent("""Por otra parte, la División Investigaciones Judiciales de la Policía Federal Argentina informó que no se dio intervención a  ninguna otra Fiscalía u otro Juzgado por la sustracción del vehículo Volkswagen Voyage, dominio KXY-876 """),
   extractions=[
            lx.data.Extraction(extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876"),
            lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL", extraction_text="Volkswagen Voyage"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""Cecilia Lopez Gracia médica del Hospital Penna, Servicio SAME, M. N. 123.558."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Cecilia Lopez Gracia"),
            lx.data.Extraction(extraction_class="NUM_MATRICULA", extraction_text="M. N. 123.558"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""A  su  vez,  requirió  informes  al  Banco BBVA   Francés, respecto  de  las  cuentas  bancarias  de  la  denunciante,  Carla Alejandra Garcia, D.N.I.  36.998.621 identificadas  como  Caja  de  ahorro  en  pesos  argentinos  número  117-59824/6  con  CBU  0180132640000004685591 y  Caja  de  ahorro  en  dólares  número  119-619018/2 con  CBU  0170115544000062081822."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Carla Alejandra Garcia"),
            lx.data.Extraction(extraction_class="DNI",         extraction_text="36.998.621"),
            lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0180132640000004685591"),
            lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="119-619018/2"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0170115544000062081822"),
        ],
    ), 

    lx.data.ExampleData(
        text=textwrap.dedent("""La grabación se encuentra disponible en el link: https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"""),
        extractions=[
            lx.data.Extraction(extraction_class="LINK", extraction_text="https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""VISTOS: Que a fin de ordenar la marcha del proceso, se fija audiencia preliminar. "
              "No se consignan números de expediente, CUIJ ni domicilios en el presente proveído."""),
        extractions=[],  # ejemplo negativo: desalienta devolver clases ausentes
    ),
]

## Functions

In [ ]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

def take_start_end_paragraphs(paragraphs):
    # Create start and end character positions
    start_end_chars = []
    current_pos = 0

    for i, paragraph in enumerate(paragraphs):
        start_char = current_pos
        end_char = start_char + len(paragraph)
        start_end_chars.append(
            {
                "paragraph_position": i,
                "text": paragraph,
                "paragraph_id": str(text_to_uuid(paragraph)).replace("-", ""),
                "start_char": start_char,
                "end_char": end_char,
            }
        )
        # +1 for the newline character between paragraphs (except after the last one)
        current_pos = end_char + 1

    return start_end_chars

def constract_paragraph(document):
    paragraphs = [line.strip() for line in document.split("\n") if line.strip()]
    paragraphs = [re.sub(r"\s{2,}", " ", line) for line in paragraphs]
    paragraphs = list(unique_justseen(paragraphs))
    return paragraphs

def langextract_to_dict(result):
    out = []
    for i in range(len(result.extractions)):
        #paragraph = result.text
        label = result.extractions[i].extraction_class
        text = result.extractions[i].extraction_text
        start_char = result.extractions[i].char_interval.start_pos
        end_char = result.extractions[i].char_interval.end_pos
        attrs = result.extractions[i].attributes
        alignment_status = result.extractions[i].alignment_status

        out.append({
            "label": label,
            "text": text,
            "start_char": start_char,
            "end_char": end_char,
            "attrs": attrs,
            "alignment_status": alignment_status
        })
        #print(f"{i+1}: \n CLASS: {result.extractions[i].extraction_class} \n TEXT: {result.extractions[i].extraction_text} \n from_chars: {text[result.extractions[i].char_interval.start_pos:result.extractions[i].char_interval.end_pos]}")
    return out

def langextract_prediction(text,PROMPT, examples,openai_api_key):
    result =  lx.extract(
        text_or_documents=text,
        prompt_description=PROMPT,
        examples=examples,
        language_model_type=lx.inference.OpenAILanguageModel,
        model_id="gpt-4o",
        api_key=openai_api_key,
        max_char_buffer=1000,
        extraction_passes=1,
        max_workers=6,
        fence_output=True,
        use_schema_constraints=False, # https://github.com/google/langextract
        language_model_params={
            "temperature": 0.1,
            "top_p": 0.9,
            "max_tokens": 400,
            "timeout": 600,},
            debug=False)
    return result, langextract_to_dict(result)

In [ ]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]

docs_file
documents = {}
doc_paragraphs = {}
doc_start_end_chars = {}
joined_texts = {}
for d in docs_file:
    path = DOCS_PATH + d
    # Extract document
    document = extraction(path)
    # Construct paragraphs
    paragraphs = constract_paragraph(document)
    start_end_chars = take_start_end_paragraphs(paragraphs)
    documents[d] = document
    doc_paragraphs[d] = paragraphs
    joined_text = "\n".join(paragraphs)
    joined_texts[d] = joined_text
    doc_start_end_chars[d] = start_end_chars


In [ ]:
paragraphs

In [ ]:
import time

start = timeit.timeit()
predictions = {}
results = {}

for name_doc, text in joined_texts.items():
    print(name_doc)
    results[name_doc], predictions[name_doc] = langextract_prediction(text, PROMPT, examples, openai_api_key)
    time.sleep(25)  # Sleep for 2 seconds to avoid rate limit

end = timeit.timeit()
print('\n Total time: ', end-start)
